## Silver Layer — Cleansed & Enriched Data

The **Silver Layer** applies structured transformations to Bronze raw data, producing a clean, typed, and enriched dataset ready for analytics and Gold aggregations.

| Transformation | Detail |
|---|---|
| Type casting | All numeric/date columns cast from `string` to `double` / `int` |
| Standardization | `Item_Fat_Content`: `LF`/`low fat` → `Low Fat`; `reg` → `Regular` |
| Null imputation | `Item_Weight` → mean imputation; `Outlet_Size` → `'Unknown'` |
| Deduplication | Natural key `(Item_Identifier, Outlet_Identifier)` — keep latest |
| Derived columns | `Outlet_Age_Years`, `Item_Visibility_Band`, `Sales_Band`, `MRP_Band` |
| Audit | `_silver_timestamp`, `_silver_processed_by`, `_bronze_ingestion_timestamp` |

In [0]:
%sql
DROP SCHEMA dev.silver CASCADE;
DROP SCHEMA dev.gold CASCADE;

In [0]:
%sql
CREATE SCHEMA dev.silver MANAGED LOCATION 'abfss://silver@devstgaccdeprj.dfs.core.windows.net/';


In [0]:
%sql
CREATE CATALOG IF NOT EXISTS dev MANAGED LOCATION 'abfss://silver@devstgaccdeprj.dfs.core.windows.net/';
CREATE SCHEMA IF NOT EXISTS dev.silver;

CREATE OR REPLACE TABLE dev.silver.sales
AS
WITH
-- Step 1: Cast all types + standardize categorical values + trivial null handling
typed AS (
    SELECT
        Item_Identifier,
        CAST(Item_Weight             AS DOUBLE)                                   AS Item_Weight,
        CASE UPPER(TRIM(Item_Fat_Content))
            WHEN 'LF'      THEN 'Low Fat'
            WHEN 'LOW FAT' THEN 'Low Fat'
            WHEN 'REG'     THEN 'Regular'
            WHEN 'REGULAR' THEN 'Regular'
            ELSE INITCAP(TRIM(Item_Fat_Content))
        END                                                                       AS Item_Fat_Content,
        CAST(Item_Visibility         AS DOUBLE)                                   AS Item_Visibility,
        TRIM(Item_Type)                                                           AS Item_Type,
        CAST(Item_MRP                AS DOUBLE)                                   AS Item_MRP,
        Outlet_Identifier,
        CAST(Outlet_Establishment_Year AS INT)                                    AS Outlet_Establishment_Year,
        COALESCE(NULLIF(TRIM(Outlet_Size), ''), 'Unknown')                        AS Outlet_Size,
        TRIM(Outlet_Location_Type)                                                AS Outlet_Location_Type,
        TRIM(Outlet_Type)                                                         AS Outlet_Type,
        CAST(Item_Outlet_Sales       AS DOUBLE)                                   AS Item_Outlet_Sales,
        _ingestion_timestamp,
        _source_filename,
        _source_system
    FROM dev.bronze.sales
),

-- Step 2: Compute mean Item_Weight for mean imputation of nulls
weight_stats AS (
    SELECT ROUND(AVG(Item_Weight), 4) AS mean_item_weight
    FROM   typed
    WHERE  Item_Weight IS NOT NULL
),

-- Step 3: Deduplicate on natural key (Item + Outlet); keep latest by ingestion time
deduped AS (
    SELECT
        t.*,
        ROW_NUMBER() OVER (
            PARTITION BY Item_Identifier, Outlet_Identifier
            ORDER BY _ingestion_timestamp DESC
        ) AS _rn
    FROM typed t
)

-- Step 4: Final Silver — impute nulls, add derived columns, audit metadata
SELECT
    d.Item_Identifier,
    ROUND(COALESCE(d.Item_Weight, w.mean_item_weight), 4)       AS Item_Weight,
    d.Item_Fat_Content,
    ROUND(d.Item_Visibility, 6)                                 AS Item_Visibility,
    d.Item_Type,
    ROUND(d.Item_MRP, 4)                                        AS Item_MRP,
    d.Outlet_Identifier,
    d.Outlet_Establishment_Year,
    d.Outlet_Size,
    d.Outlet_Location_Type,
    d.Outlet_Type,
    ROUND(d.Item_Outlet_Sales, 4)                               AS Item_Outlet_Sales,
    -- Derived: outlet age
    (YEAR(CURRENT_DATE()) - d.Outlet_Establishment_Year)        AS Outlet_Age_Years,
    -- Derived: visibility band
    CASE
        WHEN d.Item_Visibility = 0    THEN 'Not Visible'
        WHEN d.Item_Visibility < 0.05 THEN 'Low'
        WHEN d.Item_Visibility < 0.15 THEN 'Medium'
        ELSE                               'High'
    END                                                         AS Item_Visibility_Band,
    -- Derived: sales tier
    CASE
        WHEN d.Item_Outlet_Sales < 500   THEN 'Low'
        WHEN d.Item_Outlet_Sales < 2000  THEN 'Medium'
        WHEN d.Item_Outlet_Sales < 5000  THEN 'High'
        ELSE                                  'Very High'
    END                                                         AS Sales_Band,
    -- Derived: price tier
    CASE
        WHEN d.Item_MRP <  50 THEN 'Budget'
        WHEN d.Item_MRP < 100 THEN 'Economy'
        WHEN d.Item_MRP < 200 THEN 'Mid-Range'
        ELSE                       'Premium'
    END                                                         AS MRP_Band,
    -- Audit
    current_timestamp()                                         AS _silver_timestamp,
    current_user()                                              AS _silver_processed_by,
    d._ingestion_timestamp                                      AS _bronze_ingestion_timestamp,
    d._source_filename,
    d._source_system
FROM deduped d
CROSS JOIN weight_stats w
WHERE d._rn = 1;

In [0]:
bronze = spark.table("dev.bronze.sales")
silver = spark.table("dev.silver.sales")

bronze_count = bronze.count()
silver_count = silver.count()
dedup_removed = bronze_count - silver_count

print("=== Bronze → Silver Processing Summary ===")
print(f"  Bronze rows     : {bronze_count:,}")
print(f"  Silver rows     : {silver_count:,}")
print(f"  Removed (dedup) : {dedup_removed:,}")
print(f"  Bronze columns  : {len(bronze.columns)}")
print(f"  Silver columns  : {len(silver.columns)}")
print(f"  New derived cols: Outlet_Age_Years, Item_Visibility_Band, Sales_Band, MRP_Band")

In [0]:
from pyspark.sql.functions import col, count, when

df    = spark.table("dev.silver.sales")
total = df.count()
data_cols = [c for c in df.columns if not c.startswith("_")]

print(f"Post-Silver Null Analysis  (total rows: {total:,})\n")
print(f"  {'Column':<35} {'Nulls':>8}   {'%':>6}")
print("  " + "-" * 55)
any_null = False
for c in data_cols:
    n   = df.filter(col(c).isNull()).count()
    pct = n / total * 100
    flag = " ⚠️" if n > 0 else " ✅"
    if n > 0:
        any_null = True
    print(f"  {c:<35} {n:>8,}   {pct:>5.2f}%{flag}")
if not any_null:
    print("\n  ✅ No nulls remaining in any data column.")

In [0]:
%sql
SELECT
    Item_Fat_Content,
    COUNT(*)                                                  AS record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)       AS pct
FROM dev.silver.sales
GROUP BY Item_Fat_Content
ORDER BY record_count DESC

In [0]:
%sql
SELECT
    Outlet_Size,
    COUNT(*)                                                  AS record_count,
    ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2)       AS pct
FROM dev.silver.sales
GROUP BY Outlet_Size
ORDER BY record_count DESC

In [0]:
%sql
SELECT
    Sales_Band,
    MRP_Band,
    Item_Visibility_Band,
    COUNT(*)                                                  AS records,
    ROUND(AVG(Item_Outlet_Sales), 2)                          AS avg_sales,
    ROUND(AVG(Item_MRP), 2)                                   AS avg_mrp
FROM dev.silver.sales
GROUP BY Sales_Band, MRP_Band, Item_Visibility_Band
ORDER BY Sales_Band, MRP_Band

In [0]:
from pyspark.sql.functions import col

df = spark.table("dev.silver.sales")
total = df.count()

checks = {
    "No null Item_Weight (imputed)":       df.filter(col("Item_Weight").isNull()).count() == 0,
    "No null Outlet_Size (Unknown filled)": df.filter(col("Outlet_Size").isNull()).count() == 0,
    "Item_Fat_Content only Low Fat/Regular": df.filter(
        ~col("Item_Fat_Content").isin("Low Fat", "Regular")
    ).count() == 0,
    "Item_MRP > 0":                        df.filter(col("Item_MRP") <= 0).count() == 0,
    "Item_Outlet_Sales > 0":               df.filter(col("Item_Outlet_Sales") <= 0).count() == 0,
    "Item_Visibility >= 0":                df.filter(col("Item_Visibility") < 0).count() == 0,
    "Outlet_Age_Years >= 0":               df.filter(col("Outlet_Age_Years") < 0).count() == 0,
    "No duplicates (Item + Outlet key)":    df.count() == df.dropDuplicates(
        ["Item_Identifier", "Outlet_Identifier"]
    ).count(),
}

passed = sum(1 for v in checks.values() if v)
score  = passed / len(checks) * 100

print(f"=== Silver Data Quality Score: {score:.0f}%  ({passed}/{len(checks)} passed) ===\n")
for check, result in checks.items():
    status = "✅ PASS" if result else "❌ FAIL"
    print(f"  {status}   {check}")

In [0]:
%sql
DESCRIBE DETAIL dev.silver.sales